### Inisialisasi Spark Session Local


In [9]:
import sys
sys.path.append("..")

import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pathlib import Path

from nyc_taxi_etl.config import config
from nyc_taxi_etl.bronze.ingestion import stream_download_and_hash

In [10]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('nyc_taxi_eda')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)

print(f'Spark Version: {spark.version}')

Spark Version: 4.2.0


### Tarik file source


In [13]:
# 1. Siapkan direktori data lokal
local_data_dir = Path("../data/bronze")
local_data_dir.mkdir(parents=True, exist_ok=True)
local_parquet_path = local_data_dir / "yellow_tripdata_2024-01.parquet"

# 2. Unduh file ke lokal jika belum ada
if not local_parquet_path.exists():
    parquet_url = config.get_source_trip_url(year=2024, month=1)
    print(f"Mengunduh sampel data ke lokal dari: {parquet_url}")
    buffer, _, _ = stream_download_and_hash(parquet_url)
    with open(local_parquet_path, "wb") as f:
        f.write(buffer.getvalue())
    print("Selesai mengunduh ke lokal.")
else:
    print(f"File sudah ada di cache lokal: {local_parquet_path}")

# 3. Baca Parquet lokal ke PySpark DataFrame
df_raw = spark.read.parquet(str(local_parquet_path))
print(f"Total baris mentah di Bronze: {df_raw.count():,}")
df_raw.show(10, truncate=False)

File sudah ada di cache lokal: ../data/bronze/yellow_tripdata_2024-01.parquet
Total baris mentah di Bronze: 2,964,624
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |

In [ ]:
local_lookup_path = local_data_dir / "taxi_zone_lookup.csv"

if not local_lookup_path.exists():
    print("Mengunduh Taxi Zone Lookup ke lokal...")
    buffer, _, _ = stream_download_and_hash(config.zone_lookup_csv_url)
    with open(local_lookup_path, "wb") as f:
        f.write(buffer.getvalue())

df_lookup = spark.read.option("header", "true").csv(str(local_lookup_path))
print(f"Total zona: {df_lookup.count()}")
df_lookup.show(10, truncate=False)

INFO:nyc_taxi_etl.bronze.ingestion:Mengunduh file dari: https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv


Mengunduh Taxi Zone Lookup ke lokal...


INFO:nyc_taxi_etl.bronze.ingestion:Selesai mengunduh. Ukuran: 12331 bytes, SHA-256: 1a99e105092230f8620f301edcca7f80d3080642ff404d28ed957d3fa222c8ed


Total zona: 265
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10        |Queens       |Baisley Park           |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 10 rows


In [14]:
df_raw.printSchema()
df_lookup.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-

### Eksplorasi Data


In [19]:
numeric_cols = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee"
]

df_raw.select(numeric_cols).describe().show(5, truncate=False)

+-------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+
|summary|passenger_count   |trip_distance     |fare_amount       |extra             |mta_tax            |tip_amount        |tolls_amount      |improvement_surcharge|total_amount      |congestion_surcharge|airport_fee        |
+-------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+
|count  |2824462           |2964624           |2964624           |2964624           |2964624            |2964624           |2964624           |2964624              |2964624           |2824462             |2824462            |
|mean   |1.3392808966805005|3.6521691789583146|18.175061916791037|1.4515984320439959|0.483382310

### Investigasi anomali waktu


In [22]:
outside_jan_df = df_raw.filter(
    (F.col('tpep_pickup_datetime') < F.lit('2024-01-01')) |
    (F.col('tpep_pickup_datetime') >= F.lit('2024-02-01'))
)

print(f'Perjalanan di luar Januari 2024: {outside_jan_df.count():,}')

invalid_duration_df = df_raw.filter(
    (F.col('tpep_dropoff_datetime') <= F.col('tpep_pickup_datetime'))
)

print(f'Perjalanan dengan durasi negatif atau nol: {invalid_duration_df.count():,}')

Perjalanan di luar Januari 2024: 18
Perjalanan dengan durasi negatif atau nol: 870


In [24]:
# Cek distribusi payment_type untuk total_amount negatif
df_raw.filter(F.col('total_amount') < 0).groupBy('payment_type').count().show()

+------------+-----+
|payment_type|count|
+------------+-----+
|           1|   29|
|           3| 5741|
|           2| 8326|
|           4|21406|
|           0|    2|
+------------+-----+



In [26]:
# Cek kolom yang ada null

null_counts = [
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]

df_raw.select(null_counts).show(vertical=True, truncate=False)

-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 140162 
 trip_distance         | 0      
 RatecodeID            | 140162 
 store_and_fwd_flag    | 140162 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 140162 
 Airport_fee           | 140162 



In [28]:
# Konversi LocationID lookup ke integer
lookup_ids = df_lookup.select(F.col('LocationID').cast(T.IntegerType()).alias('loc_id'))

# Cari PULocationID yang tidak ada di lookup
unknown_pu = df_raw.join(lookup_ids, df_raw.PULocationID == lookup_ids.loc_id, how='left_anti')
print(f'PULocationID yang tidak dikenali di lookup: {unknown_pu.count():,}')

#Cari DOLocationID yang tidak ada di lookup
unknown_do = df_raw.join(lookup_ids, df_raw.DOLocationID == lookup_ids.loc_id, how='left_anti')
print(f'DOLocationID yang tidak dikenali di lookup: {unknown_do.count():,}')


PULocationID yang tidak dikenali di lookup: 0
DOLocationID yang tidak dikenali di lookup: 0
